<a href="https://colab.research.google.com/github/dechanel97/End-to-End-Data-Warehouse-Power-BI-Project/blob/main/FintuningFinancial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ÉTAPE 1 — Installer l'environnement (chapitre 1.3)

In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install transformers peft trl accelerate bitsandbytes datasets evaluate
!pip install streamlit plotly pandas numpy yfinance
#!pip install MetaTrader5          # uniquement en local Windows
#!pip install coinbase-advanced-py python-binance ccxt   # pour le chapitre 16

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 123.0 MB/s eta 0:00:00


In [2]:
!pip install feedparser
import importlib
import elephantmind_rss_collector

importlib.reload(elephantmind_rss_collector)

elephantmind_rss_collector.build_rss_dataset()

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.8 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=167a68c91209da85c75de832fd089199fb42463ce83cbd04a68cce369cd53e5c
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
[ok] EURUSD=X <- 'EUR USD forex news' : 15 articles
[ok] EURUSD=X <- 'ECB interest rates' : 15 articles
[ok] EURUSD=X <- 'Federal Reserve interest rates' : 15 articles
[ok] EURUSD=X <- 'EURUSD market analysis' : 15 articles
[ok] BTC-USD <- 'Bitcoin news' : 15 articles
[ok] BTC-USD <- 'BTC market analysis' : 15 articles
[ok] BTC-USD <- 'cryptocurrency market' : 15 articles
[ok] BTC-USD <- 'Bitcoin regulation' : 15 articles
[ok] ^GSPC  <- 'S&P 500 news' : 15 articles
[ok] ^GSPC  <- 'US stock market' : 15 articles
[ok] ^GSPC  <- 'Wall Street market analysis' : 15 articles
[ok] ^GSPC  <-

,time,symbol,text,source,link
157,2026-04-01 07:00:00,^GSPC,Watch Markets Analysis: Equities Surge Could U...,Bloomberg.com,https://news.google.com/rss/articles/CBMirgFBV...
39,2026-05-18 07:00:00,EURUSD=X,The Fed will have to raise interest rates in J...,CNBC,https://news.google.com/rss/articles/CBMixAFBV...
37,2026-05-20 07:00:00,EURUSD=X,"Fed interest-rate rate hike chances grow, minu...",MarketWatch,https://news.google.com/rss/articles/CBMioAFBV...
36,2026-05-20 07:00:00,EURUSD=X,Fed officials’ concerns about inflation sparke...,NBC News,https://news.google.com/rss/articles/CBMiqwFBV...
161,2026-05-23 07:00:00,^GSPC,Stock market gains on Wall Street close: S&P 5...,eciks.org,https://news.google.com/rss/articles/CBMirgFBV...
...,...,...,...,...,...
110,2026-07-11 19:42:04,BTC-USD,Ripple wins Europe before XRP wins legal clari...,Crypto News,https://news.google.com/rss/articles/CBMif0FVX...
176,2026-07-11 19:48:07,^GSPC,What Trump Voters Say About the Economy Now - WSJ,WSJ,https://news.google.com/rss/articles/CBMib0FVX...
102,2026-07-11 20:11:09,BTC-USD,"SOL price on Jul 12, 2026 at 5pm EDT Crypto Pr...",Robinhood,https://news.google.com/rss/articles/CBMirgFBV...
62,2026-07-11 20:27:00,BTC-USD,Oil Is Below $76. So Why Is Bitcoin Still Belo...,Yahoo Finance,https://news.google.com/rss/articles/CBMikwFBV...


## ÉTAPE 2 — Récupérer les données de marché (chapitre 6.4)

In [3]:
# elephantmind_yfinance_pipeline.py
import time
import numpy as np
import pandas as pd
import yfinance as yf

SYMBOLS = {"EURUSD=X": "EURUSD", "BTC-USD": "BTCUSD", "^GSPC": "SPX500"}
INTERVAL = "1h"
OUTPUT_PARQUET = "dataset_ohlcv_features.parquet"
NEWS_INPUT_PATH = "dataset_x_supply_chain_sentiment.parquet"

def download_with_retry(ticker, start=None, end=None, interval='1h', max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            df = yf.download(ticker, start=start, end=end, interval=interval,
                              progress=False, auto_adjust=True, multi_level_index=False)
            if df is not None and not df.empty:
                return df
        except Exception as e:
            print(f"[{ticker}] erreur: {e} (tentative {attempt}/{max_retries})")
        time.sleep(2 * attempt)
    raise RuntimeError(f"Impossible de recuperer {ticker}")

def fetch_all_symbols(symbols, start_date, end_date, interval):
    data = {}
    for yf_ticker, name in symbols.items():
        df = download_with_retry(yf_ticker, start=start_date, end=end_date, interval=interval)
        df.columns = [c.lower() for c in df.columns]
        df.index.name = "time"
        data[name] = df
        time.sleep(1)
    return data

def compute_rsi(close, period=14):
    delta = close.diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    rs = gain.rolling(period).mean() / loss.rolling(period).mean().replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(close, fast=12, slow=26, signal=9):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    return macd_line, macd_line.ewm(span=signal, adjust=False).mean()

def compute_volatility(close, window=20):
    return np.log(close / close.shift(1)).rolling(window).std() * np.sqrt(252)

def add_features(df):
    df = df.copy()
    df["rsi"] = compute_rsi(df["close"])
    df["macd"], df["macd_signal"] = compute_macd(df["close"])
    df["volatility_20"] = compute_volatility(df["close"])
    df["sma_50"] = df["close"].rolling(50).mean()
    df["above_sma50"] = (df["close"] > df["sma_50"]).astype(int)
    return df.dropna()

def build_context_row(symbol, row):
    macd_trend = "haussier" if row["macd"] > row["macd_signal"] else "baissier"
    trend = "au-dessus" if row["above_sma50"] else "en-dessous"
    return (f"{symbol}: prix={row['close']:.5f}, RSI={row['rsi']:.1f}, "
            f"MACD={macd_trend}, prix {trend} de la SMA50, "
            f"volatilite annualisee={row['volatility_20']:.2%}.")

def main():
    # Load news to determine date range for OHLCV data
    news_df = pd.read_parquet(NEWS_INPUT_PATH)
    # Add a buffer to ensure full coverage and for feature calculation
    min_news_date = news_df['time'].min() - pd.Timedelta(days=5)
    max_news_date = news_df['time'].max() + pd.Timedelta(days=5)

    raw_data = fetch_all_symbols(SYMBOLS, min_news_date, max_news_date, INTERVAL)
    all_rows = []
    for symbol, df in raw_data.items():
        df_feat = add_features(df)
        df_feat["symbol"] = symbol
        df_feat["context_text"] = df_feat.apply(lambda r: build_context_row(symbol, r), axis=1)
        all_rows.append(df_feat.reset_index())
    dataset = pd.concat(all_rows, ignore_index=True)
    dataset.to_parquet(OUTPUT_PARQUET, index=False)

if __name__ == "__main__":
    main()

## ÉTAPE 2bis — Construire les labels de news (nouveau, prérequis de l'étape 3)

In [4]:
# elephantmind_news_labels_builder.py
import pandas as pd
import pytz

# METHODE B (rapide) : auto-labelling par le rendement futur du prix.
# Le label suppose que le mouvement de prix est cause par la news/le tweet —
# approximation utile pour du volume, a valider en backtest (chapitre 8.3)
# avant de faire confiance au signal.

def auto_label_from_price_move(
    news_path, ohlcv_path="dataset_ohlcv_features.parquet",
    horizon_hours=6, move_threshold=0.005,
    output_parquet="dataset_news_labels.parquet",
):
    news = pd.read_parquet(news_path).sort_values("time")
    ohlcv = pd.read_parquet(ohlcv_path).sort_values("time")

    # Define the symbol mapping as used in elephantmind_yfinance_pipeline.py
    symbol_map = {"^GSPC": "SPX500", "EURUSD=X": "EURUSD", "BTC-USD": "BTCUSD"}

    # Normalize news symbols to match OHLCV symbols
    news['symbol'] = news['symbol'].replace(symbol_map)

    # Ensure 'time' columns are timezone-aware (UTC) for consistent comparison
    # Convert news 'time' to UTC if it's naive
    if news['time'].dtype == 'datetime64[ns]':
        news['time'] = news['time'].dt.tz_localize(pytz.utc) # Assume naive timestamps are UTC
    else:
        news['time'] = news['time'].dt.tz_convert(pytz.utc)

    # Convert ohlcv 'time' to UTC if it's not already or ensure it's consistent
    if ohlcv['time'].dtype == 'datetime64[ns]': # Check if it's naive first
        ohlcv['time'] = ohlcv['time'].dt.tz_localize(pytz.utc)
    else: # Already tz-aware, just convert to UTC if different
        ohlcv['time'] = ohlcv['time'].dt.tz_convert(pytz.utc)

    rows = []
    for symbol_ohlcv, group in ohlcv.groupby("symbol"):
        group = group.set_index("time")["close"]
        news_symbol_filtered = news[news["symbol"] == symbol_ohlcv]

        for _, n in news_symbol_filtered.iterrows():
            t0 = n["time"]
            t1 = t0 + pd.Timedelta(hours=horizon_hours)

            # asof requires monotonic index
            price_before = group.asof(t0)
            price_after = group.asof(t1)

            if pd.isna(price_before) or pd.isna(price_after):
                continue

            forward_return = (price_after - price_before) / price_before
            if forward_return > move_threshold:
                signal, reason = "BUY", f"Prix en hausse de {forward_return:.2%} dans les {horizon_hours}h suivantes"
            elif forward_return < -move_threshold:
                signal, reason = "SELL", f"Prix en baisse de {forward_return:.2%} dans les {horizon_hours}h suivantes"
            else:
                signal, reason = "HOLD", f"Mouvement de prix negligeable ({forward_return:.2%})"

            confidence = min(abs(forward_return) / (move_threshold * 4), 1.0)
            rows.append({
                "time": t0, "symbol": symbol_ohlcv,
                "news_summary": n.get("text", n.get("news_summary", "")),
                "label_signal": signal, "label_confidence": round(confidence, 2),
                "label_reason": reason,
            })

    df = pd.DataFrame(rows)
    df.to_parquet(output_parquet, index=False)
    print(f"{len(df)} lignes auto-labellisees exportees vers {output_parquet}")

if __name__ == "__main__":
    # Exemple : a partir des tweets deja collectes (etape 3bis)
    auto_label_from_price_move(news_path="dataset_x_supply_chain_sentiment.parquet")

175 lignes auto-labellisees exportees vers dataset_news_labels.parquet


## ÉTAPE 3 — Fusionner OHLCV + news → dataset d'entraînement (chapitre 6.4.4)

In [5]:
import pandas as pd
import json

ohlcv = pd.read_parquet("dataset_ohlcv_features.parquet").sort_values("time")
news = pd.read_parquet("dataset_news_labels.parquet").sort_values("time")

merged = pd.merge_asof(
    ohlcv, news, on="time", by="symbol",
    direction="backward", tolerance=pd.Timedelta("2h"),
)
merged = merged.dropna(subset=["label_signal"])

with open("train.jsonl", "w", encoding="utf-8") as f:
    for _, row in merged.iterrows():
        full_context = f"{row['context_text']} News: {row['news_summary']}"
        example = {
            "instruction": "Analyse le contexte de marche suivant et donne un signal de trading.",
            "input": full_context,
            "output": json.dumps({
                "signal": row["label_signal"],
                "confidence": float(row["label_confidence"]),
                "reason": row["label_reason"],
            }, ensure_ascii=False),
        }
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

## ÉTAPE 3bis — Option B : Collecter des news sectorielles via RSS Google News (sans clé, gratuit)

In [6]:
# elephantmind_rss_collector.py
import time as time_module
from datetime import datetime, timezone

import feedparser
import pandas as pd

# Un ticker -> une ou plusieurs requetes de recherche (mots-cles pertinents
# pour la chaine d'approvisionnement / actualite du secteur).
TICKER_QUERIES = {
    "EURUSD=X": [
        "EUR USD forex news",
        "ECB interest rates",
        "Federal Reserve interest rates",
        "EURUSD market analysis"
    ],

    "BTC-USD": [
        "Bitcoin news",
        "BTC market analysis",
        "cryptocurrency market",
        "Bitcoin regulation"
    ],

    "^GSPC": [
        "S&P 500 news",
        "US stock market",
        "Wall Street market analysis",
        "US economy"
    ],
}

GOOGLE_NEWS_RSS = "https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"


def _fetch_one(query: str, symbol: str, max_items: int = 15) -> list[dict]:
    url = GOOGLE_NEWS_RSS.format(query=query.replace(" ", "+"))
    feed = feedparser.parse(url)
    rows = []
    for entry in feed.entries[:max_items]:
        if getattr(entry, "published_parsed", None):
            dt = datetime.fromtimestamp(
                time_module.mktime(entry.published_parsed), tz=timezone.utc
            )
        else:
            dt = datetime.now(tz=timezone.utc)

        rows.append(
            {
                "time": dt.replace(tzinfo=None),
                "symbol": symbol,
                "text": entry.title,
                "source": getattr(entry, "source", {}).get("title", "google_news")
                if hasattr(entry, "source")
                else "google_news",
                "link": entry.link,
            }
        )
    return rows


def build_rss_dataset(
    tickers: dict[str, list[str]] = None,
    max_items_per_query: int = 15,
    out_path: str = "dataset_x_supply_chain_sentiment.parquet",
) -> pd.DataFrame:
    tickers = tickers or TICKER_QUERIES
    all_rows = []

    for symbol, queries in tickers.items():
        for query in queries:
            try:
                rows = _fetch_one(query, symbol, max_items=max_items_per_query)
                all_rows.extend(rows)
                print(f"[ok] {symbol:6s} <- '{query}' : {len(rows)} articles")
            except Exception as e:
                print(f"[erreur] {symbol:6s} <- '{query}' : {e}")
            time_module.sleep(1)  # petite pause polie entre les requetes

    if not all_rows:
        raise RuntimeError(
            "Aucun article recupere. Verifie ta connexion internet."
        )

    df = pd.DataFrame(all_rows)
    df = df.drop_duplicates(subset=["text", "symbol"]).sort_values("time")
    df.to_parquet(out_path, index=False)

    print(f"\n{len(df)} lignes ecrites dans {out_path}")
    print(df["symbol"].value_counts())
    return df


if __name__ == "__main__":
    build_rss_dataset()

[ok] EURUSD=X <- 'EUR USD forex news' : 15 articles
[ok] EURUSD=X <- 'ECB interest rates' : 15 articles
[ok] EURUSD=X <- 'Federal Reserve interest rates' : 15 articles
[ok] EURUSD=X <- 'EURUSD market analysis' : 15 articles
[ok] BTC-USD <- 'Bitcoin news' : 15 articles
[ok] BTC-USD <- 'BTC market analysis' : 15 articles
[ok] BTC-USD <- 'cryptocurrency market' : 15 articles
[ok] BTC-USD <- 'Bitcoin regulation' : 15 articles
[ok] ^GSPC  <- 'S&P 500 news' : 15 articles
[ok] ^GSPC  <- 'US stock market' : 15 articles
[ok] ^GSPC  <- 'Wall Street market analysis' : 15 articles
[ok] ^GSPC  <- 'US economy' : 15 articles

176 lignes ecrites dans dataset_x_supply_chain_sentiment.parquet
symbol
^GSPC       60
BTC-USD     59
EURUSD=X    57
Name: count, dtype: int64


## ÉTAPE 4 — Charger le modèle de base en QLoRA (chapitres 5.3, 7.1, 13.1, 13.2)

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#model_name = "Qwen/Qwen2.5-7B-Instruct"
#model_name = "Qwen/Qwen3.6-27B"
#model_name = "Qwen/Qwen3-8B"
model_name = "Qwen/Qwen3-1.7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # entrainement toujours en bf16 (13.1)
    bnb_4bit_use_double_quant=True,
    # Removed llm_int8_enable_fp32_cpu_offload as no GPU is available
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Optionnel (13.2) : tokens speciaux pour structurer le contexte
new_tokens = ["<SIGNAL>", "<MACRO>", "<RISK", "<PORTFOLIO>"]
tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})

# Removed max_memory_mapping as no GPU is available

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    attn_implementation="sdpa",      # 13.7
    device_map="cpu", # Explicitly set to CPU as no accelerator device is available
    # Removed max_memory as it's not needed for explicit CPU loading without GPU
)
model.resize_token_embeddings(len(tokenizer))
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],           # elargir a k_proj/o_proj si besoin (13.2)
    modules_to_save=["embed_tokens", "lm_head"],    # necessaire car vocabulaire modifie
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 624,463,872 || all params: 2,344,500,224 || trainable%: 26.6353


## ÉTAPE 5 — Préparer le dataset avec le chat template (chapitres 7.2, 13.4)

In [8]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl", split="train")

SYSTEM_PROMPT = (
    "Tu es un analyste de marche pour un systeme de trading automatise. "
    "Reponds uniquement avec un objet JSON valide contenant les cles "
    "'signal' (BUY, SELL ou HOLD), 'confidence' (0 a 1) et 'reason' (une phrase)."
)

def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/179 [00:00<?, ? examples/s]

## ÉTAPE 6 — Entraîner avec SFTTrainer (chapitres 7.3, 13.5, 13.6)

In [12]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./elephantmind-lora-checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    # max_seq_length=1024, # Moved to SFTTrainer (but removed in next step)
    dataset_text_field="text",
    packing=True,                 # 13.6 : accelere l'entrainement sur contextes courts
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    # Removed tokenizer=tokenizer as it's not expected by SFTTrainer
    # Removed max_seq_length as it's not expected by SFTTrainer
    # completion_only_loss=True   # equivalent TRL du principe explique en 13.5
)

trainer.train()
trainer.save_model("./elephantmind-lora-final")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/179 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/179 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/179 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/179 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


## ÉTAPE 7 — Fusionner l'adaptateur pour le déploiement (chapitre 7.5)

In [ ]:
!pip install --upgrade torchao
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer # Import AutoTokenizer as it's used
from peft import PeftModel

# Clear CUDA cache if any previous operations left memory on GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load the base model without BitsAndBytesConfig for merging to ensure it's fully on CPU
# The model_name and tokenizer should be defined in a previous cell (ocgVz4CYZX2i)
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="cpu")

# Re-apply the tokenizer special tokens and resize embeddings to match the trained PEFT model
# new_tokens definition must be accessible (e.g., from a previous cell or redefined here if needed)
# For simplicity, assuming 'new_tokens' is defined in the global scope from ocgVz4CYZX2i
tokenizer.add_special_tokens({"additional_special_tokens": new_tokens}) # Ensure tokenizer is up-to-date
base_model.resize_token_embeddings(len(tokenizer))

# Temporarily disable CUDA visibility to ensure PEFT model loading is forced to CPU
original_cuda_visible_devices = os.environ.get("CUDA_VISIBLE_DEVICES", None)
os.environ["CUDA_VISIBLE_DEVICES"] = ""

try:
    # Explicitly load adapter to CPU, adding low_cpu_mem_usage=True as a potential helper
    peft_model = PeftModel.from_pretrained(base_model, "./elephantmind-lora-final", device_map='cpu', low_cpu_mem_usage=True)
finally:
    # Restore CUDA_VISIBLE_DEVICES
    if original_cuda_visible_devices is not None:
        os.environ["CUDA_VISIBLE_DEVICES"] = original_cuda_visible_devices
    else:
        del os.environ["CUDA_VISIBLE_DEVICES"]

merged_model = peft_model.merge_and_unload()
merged_model = merged_model.to('cpu') # Explicitly move merged model to CPU after merging
merged_model.save_pretrained("./elephantmind-merged")
tokenizer.save_pretrained("./elephantmind-merged")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pour tester le format de sortie, nous devons d'abord générer quelques prédictions à l'aide du modèle fusionné (`merged_model`). Nous allons prendre un sous-ensemble des données `merged` comme entrées.

In [19]:
from tqdm.notebook import tqdm

# Prepare inputs for the model
# We'll use a small subset of the merged data for demonstration
# Let's take the first 10 examples from the merged dataset for testing
test_examples = merged.head(10).apply(
    lambda row: {'input': f"{row['context_text']} News: {row['news_summary']}"},
    axis=1
).tolist()

# Format the inputs using the chat template (similar to what was done for training)
def format_inference_input(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

inference_dataset = dataset.select(range(10)).map(format_inference_input)

# Generate predictions
model_outputs = []
for i, example in enumerate(tqdm(inference_dataset, desc="Generating predictions")):
    inputs = tokenizer(example['text'], return_tensors="pt").to(base_model.device) # Ensure inputs are on the correct device

    # Generate output with a maximum length to prevent infinite generation
    # Set appropriate generation parameters like max_new_tokens, do_sample, top_p, etc.
    output_tokens = merged_model.generate(
        **inputs,
        max_new_tokens=100, # Adjust as needed for your expected output length
        do_sample=True, # Enable sampling for more varied outputs
        top_p=0.9,      # Nucleus sampling
        temperature=0.7, # Sampling temperature
        pad_token_id=tokenizer.eos_token_id # Important for handling padding in generation
    )

    # Decode the generated tokens
    # We want only the newly generated text, not the input prompt
    generated_text = tokenizer.decode(output_tokens[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    model_outputs.append(generated_text)

print("Premières sorties générées :")
for i, output in enumerate(model_outputs[:3]): # Display first 3 outputs
    print(f"Output {i+1}: {output}")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Generating predictions:   0%|          | 0/10 [00:00<?, ?it/s]

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## ÉTAPE 8 — Valider le format de sortie avant tout backtest (chapitre 8.2)

In [18]:
import json

def is_valid_output(text):
    try:
        obj = json.loads(text)
        assert obj["signal"] in ["BUY", "SELL", "HOLD"]
        assert 0.0 <= obj["confidence"] <= 1.0
        assert isinstance(obj["reason"], str) and len(obj["reason"]) > 0
        return True
    except Exception:
        return False

valid = sum(is_valid_output(o) for o in model_outputs)
print(f"Taux de sorties valides: {valid / len(model_outputs):.2%}")

NameError: name 'model_outputs' is not defined